## Sezione 1 — Setup & Imports

Stack tecnologico: LangGraph per orchestrazione agenti, Groq (LPU inference) 
come LLM backbone tramite `langchain-groq`, pandas/numpy per manipolazione dati.

Il modello scelto è `llama-3.3-70b-versatile`: qualità di reasoning paragonabile 
a GPT-4, free tier Groq con 30 RPM e 14.400 richieste/giorno — sufficiente per 
l'intera pipeline multi-agente senza rate limit issues.

In [1]:
import os, json
import pandas as pd
import numpy as np
from typing import TypedDict, Optional, Annotated
import operator
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, END

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

# Verifica operativa — fallisce esplicitamente se la chiave non è valida
response = llm.invoke("Reply with exactly one word: OK")
assert "OK" in response.content, f"Risposta inattesa: {response.content}"
print(f"Groq API: OK — modello: {llm.model_name}")

Groq API: OK — modello: llama-3.3-70b-versatile


## Phase 2 — Data Exploration

Before building any agent or quality tool, we must understand the raw state of each dataset. 
This phase profiles all four CSVs along four dimensions: **schema structure**, **completeness**, 
**format consistency**, and **disguised null patterns**. The anomalies discovered here directly 
motivate every deterministic tool built in Phase 3.

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spesa = pd.read_csv("data/project_data_quality/spesa.csv")
attivazioni = pd.read_csv("data/project_data_quality/attivazioniCessazioni.csv")
tipologia = pd.read_csv("data/project_anomaly_detection/TIPOLOGIA_VIAGGIATORE.csv")
allarmi = pd.read_csv("data/project_anomaly_detection/ALLARMI.csv")

datasets = {
    "spesa": spesa,
    "attivazioni": attivazioni,
    "tipologia": tipologia,
    "allarmi": allarmi
}

for name, df in datasets.items():
    print(f"[{name}]  rows={df.shape[0]}  columns={df.shape[1]}")

[spesa]  rows=7543  columns=18
[attivazioni]  rows=20102  columns=19
[tipologia]  rows=5095  columns=33
[allarmi]  rows=5080  columns=24


### 2.1 — Schema Audit: Column Naming Conventions

We inspect all column names for violations of standard naming conventions. 
A well-formed schema uses lowercase snake_case, avoids special characters, 
and never starts with a digit. Violations here will break downstream 
SQL ingestion, pandas attribute access, and API serialization.

In [3]:
import re
import pandas as pd

def audit_schema(df, name):
    issues = []
    for col in df.columns:
        flags = []
        if col != col.lower():
            flags.append("mixed_case")
        if re.search(r'[%\s-]', col):  # hyphen at end of class
            flags.append("special_char_or_space")
        if re.match(r'^\d', col):
            flags.append("starts_with_digit")
        if flags:
            issues.append({"dataset": name, "column": col, "violations": ", ".join(flags)})
    return pd.DataFrame(issues)

results = [audit_schema(df, name) for name, df in datasets.items()]
schema_issues = pd.concat(results, ignore_index=True) if results else pd.DataFrame()

if schema_issues.empty:
    print("No schema issues found.")
else:
    print(schema_issues.to_string(index=False))

    dataset                column                        violations
      spesa      aggregation-time             special_char_or_space
      spesa          Tipo Imposta mixed_case, special_char_or_space
      spesa          SPESA TOTALE mixed_case, special_char_or_space
      spesa          2cod_imposta                 starts_with_digit
      spesa       cod imposta ext             special_char_or_space
      spesa             ente%code             special_char_or_space
attivazioni                  RATA                        mixed_case
attivazioni      aggregation-time             special_char_or_space
attivazioni        Provincia Sede mixed_case, special_char_or_space
attivazioni           CODICE ENTE mixed_case, special_char_or_space
attivazioni          3descrizione                 starts_with_digit
attivazioni          regione%sede             special_char_or_space
attivazioni          att ivazioni             special_char_or_space
  tipologia           NAZIONALITA               

# completeness profile

In [4]:
def completeness_profile(df, name):
    DISGUISED_NULLS = {"n.d.", "nd", "n/a", "null", "none", "?", "-", "//", 
                       "unknown", "na", "missing", "", " "}
    
    result = []
    for col in df.columns:
        total = len(df)
        real_nulls = df[col].isna().sum()
        
        str_col = df[col].astype(str).str.strip().str.lower()
        disguised = str_col.isin(DISGUISED_NULLS).sum()
        
        effective_missing = real_nulls + disguised
        completeness_pct = round(100 * (1 - effective_missing / total), 2)
        
        result.append({
            "dataset": name, "column": col,
            "null_count": real_nulls,
            "disguised_null_count": disguised,
            "completeness_%": completeness_pct
        })
    return pd.DataFrame(result)

completeness = pd.concat([completeness_profile(df, name) for name, df in datasets.items()])

# Show the most problematic columns
sparse = completeness[completeness["completeness_%"] < 85].sort_values("completeness_%")
print(sparse[["dataset","column","null_count","disguised_null_count","completeness_%"]].to_string(index=False))

    dataset          column  null_count  disguised_null_count  completeness_%
attivazioni      fonte_dato       19942                     0            0.80
  tipologia  codice_rischio        5054                     0            0.80
      spesa      fonte_dato        7468                     0            0.99
    allarmi    flag_rischio        5029                     0            1.00
  tipologia  note_operatore        5034                     0            1.20
attivazioni            note       19802                     0            1.49
    allarmi  note_operatore        5003                     0            1.52
      spesa            note        7393                     0            1.99
  tipologia  TIPO_DOCUMENTO          62                  1748           64.47
  tipologia  Tipo Documento           0                  1675           67.12
attivazioni       qualifica        5086                     0           74.70
  tipologia ESITO_CONTROLLO        1289                     0   

# format consistency detection

In [5]:
# Focus on date/temporal columns and key categorical columns

def sample_unique_values(df, col, n=8):
    return df[col].dropna().astype(str).str.strip().unique()[:n].tolist()

# --- DATE FORMAT AUDIT ---
date_cols = {
    "spesa":       ["rata", "aggregation-time"],
    "attivazioni": ["mese", "anno", "RATA", "aggregation-time"],
    "tipologia":   ["DATA_PARTENZA", "ANNO_PARTENZA"],
    "allarmi":     ["DATA_PARTENZA", "ANNO_PARTENZA"]
}

for ds_name, cols in date_cols.items():
    df = datasets[ds_name]
    print(f"\n=== {ds_name.upper()} ===")
    for col in cols:
        if col in df.columns:
            samples = sample_unique_values(df, col)
            print(f"  [{col}]: {samples}")


=== SPESA ===
  [rata]: ['202402', '202406', '202408', '202404', '202410', '202405', '202409', '202403']
  [aggregation-time]: ['2024-03-11T02:01:04.421', '2024-07-11T03:01:16.866', '2024-09-11T03:01:11.704', '2024-05-11T03:01:07.269', '2024-11-11T02:00:28.485', '2024-06-11T03:01:23.745', '2024-10-24T18:30:35.560', '2024/04/11']

=== ATTIVAZIONI ===
  [mese]: ['11', '7', '8', '4', '6', '2', '1', '12']
  [anno]: ['2023', '2021', '2024', '23', '2024.', '2022', '2023.', 'anno 2024']
  [RATA]: ['202311', '202307', '202308', '202304', '202306', '202402', '202301', '202401']
  [aggregation-time]: ['2025-06-18T16:15:20.148346', 'GIU 18 2025', '18/06/2025', '2025/06/18', '18.06.2025', '18-06-25']

=== TIPOLOGIA ===
  [DATA_PARTENZA]: ['2024-02-13 07:30:00', '2024-01-22 16:35:00', '2024-02-04 20:10:00', '2024-01-25 13:05:00', 'FEB 13 2024', '2024-02-18 16:30:00', '2024-01-19 07:15:00', '2024-01-11 21:55:00']
  [ANNO_PARTENZA]: ['2024', '2023', '24', 'anno 2024']

=== ALLARMI ===
  [DATA_PARTEN

# outliers and categorical anomaly scan

In [6]:
# Numerical outlier hints + impossible values
print("=== NUMERICAL ANOMALIES ===")

# spesa: spesa column with currency symbol
spesa_currency = spesa[spesa['spesa'].astype(str).str.contains('€', na=False)]
print(f"[spesa] Rows with '€' in `spesa` column: {len(spesa_currency)}")

# allarmi: negative TOT value
allarmi_neg = allarmi[pd.to_numeric(allarmi['TOT'], errors='coerce') < 0]
print(f"[allarmi] Rows with negative TOT: {len(allarmi_neg)}")

# allarmi: TOT as non-numeric string
allarmi_bad_tot = allarmi[pd.to_numeric(allarmi['TOT'], errors='coerce').isna() & allarmi['TOT'].notna()]
print(f"[allarmi] Rows with non-numeric TOT (e.g. '~1', 'N.D.', '1 voli'): {len(allarmi_bad_tot)}")
print(f"  Samples: {allarmi_bad_tot['TOT'].unique()[:6].tolist()}")

# attivazioni: extreme attivazioni/cessazioni values
att_col = pd.to_numeric(attivazioni['attivazioni'], errors='coerce')
print(f"\n[attivazioni] `attivazioni` — max={att_col.max()}, 99th pct={att_col.quantile(0.99):.0f}")
ces_col = pd.to_numeric(attivazioni['cessazioni'], errors='coerce')
print(f"[attivazioni] `cessazioni` — max={ces_col.max()}, 99th pct={ces_col.quantile(0.99):.0f}")

# tipologia: INVESTIGATI > ENTRATI (logical impossibility)
tip = tipologia.copy()
tip['ENTRATI_n']    = pd.to_numeric(tip['ENTRATI'],    errors='coerce')
tip['INVESTIGATI_n']= pd.to_numeric(tip['INVESTIGATI'],errors='coerce')
cross_issue = tip[tip['INVESTIGATI_n'] > tip['ENTRATI_n']]
print(f"\n[tipologia] Rows where INVESTIGATI > ENTRATI (logical violation): {len(cross_issue)}")

=== NUMERICAL ANOMALIES ===
[spesa] Rows with '€' in `spesa` column: 56
[allarmi] Rows with negative TOT: 7
[allarmi] Rows with non-numeric TOT (e.g. '~1', 'N.D.', '1 voli'): 156
  Samples: ['N.D.', '?', '~181', '1 voli', '~1', '~167']

[attivazioni] `attivazioni` — max=36423.0, 99th pct=2483
[attivazioni] `cessazioni` — max=28566.0, 99th pct=2640

[tipologia] Rows where INVESTIGATI > ENTRATI (logical violation): 216


## Phase 3 — Deterministic Tool Library

In this phase we convert the exploratory findings from Phase 2 into a deterministic validation layer.  
Each function acts like a strict audit tool: it returns structured evidence, severity, and remediation hints instead of free-form text.

These tools will later be exposed to the LangGraph agents so the LLM can reason on top of hard evidence rather than guessing from raw CSV contents.

In [8]:
import re
import json
import numpy as np
import pandas as pd
from collections import defaultdict

DISGUISED_NULLS = {
    "n.d.", "nd", "n/a", "null", "none", "?", "-", "//",
    "unknown", "na", "missing", "", " "
}

SEVERITY_RANK = {"low": 1, "medium": 2, "high": 3, "critical": 4}

# Strips surrounding whitespace and returns None for empty strings.
def normalize_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x != "" else None

# Lower-cases a value after normalizing it; used when comparing tokens case-insensitively.
def normalize_token(x):
    x = normalize_text(x)
    return x.lower() if x is not None else None

# Returns True when a value is semantically empty (placeholder string) even if pandas sees it as non-null.
def is_disguised_null(x):
    token = normalize_token(x)
    return token in DISGUISED_NULLS if token is not None else False

# Pulls up to n distinct non-null string samples from a series for use as evidence in issue reports.
def safe_samples(series, n=5):
    s = series.dropna().astype(str).str.strip()
    s = s[s.ne("")]
    return s.drop_duplicates().head(n).tolist()

# Attempts a best-effort numeric parse by stripping currency symbols, spaces,
# thousand-dot separators, and comma decimals before calling pd.to_numeric.
def coerce_numeric_loose(series):
    s = series.astype(str).str.strip()
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan, "none": np.nan})
    s = s.str.replace("€", "", regex=False)
    s = s.str.replace(r"\s+", "", regex=True)
    s = s.str.replace(r"(?<=\d)\.(?=\d{3}(\D|$))", "", regex=True)
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    s = s.replace({"": np.nan, "-": np.nan, ".": np.nan, "-.": np.nan})
    return pd.to_numeric(s, errors="coerce")

# Guesses whether a column is "numeric", "date", or "string" based on
# what fraction of present values can be parsed as each type.
def infer_semantic_type(series, threshold=0.90):
    s = series.copy()
    mask_present = s.notna() & s.astype(str).str.strip().ne("")
    s = s[mask_present]
    if len(s) == 0:
        return "unknown"
    numeric_rate = coerce_numeric_loose(s).notna().mean()
    date_rate = pd.to_datetime(s.astype(str).str.strip(), errors="coerce", dayfirst=True).notna().mean()
    if numeric_rate >= threshold:
        return "numeric"
    if date_rate >= threshold:
        return "date"
    return "string"

# Builds a single issue dict with a consistent schema so all tools speak the same language
# and LangGraph agents can parse results without any special-casing.
def make_issue(dataset, tool, issue_type, severity, message,
               columns=None, row_count=None, evidence=None, suggested_fix=None):
    return {
        "dataset": dataset,
        "tool": tool,
        "issue_type": issue_type,
        "severity": severity,
        "columns": columns if columns is not None else [],
        "row_count": int(row_count) if row_count is not None else None,
        "message": message,
        "evidence": evidence if evidence is not None else {},
        "suggested_fix": suggested_fix
    }

# Wraps a list of issues from one tool into a standard result envelope that includes
# an issue count, a per-severity breakdown, and optional metadata.
def build_result(dataset, tool, issues, meta=None):
    severity_breakdown = defaultdict(int)
    for issue in issues:
        severity_breakdown[issue["severity"]] += 1
    return {
        "dataset": dataset,
        "tool": tool,
        "issue_count": len(issues),
        "severity_breakdown": dict(severity_breakdown),
        "issues": issues,
        "meta": meta or {}
    }

## 3.1 — Expected Schema Registry

A schema tool should never hallucinate what the dataset “should” look like.  
So we first seed a baseline schema directly from the currently loaded DataFrames, then allow manual overrides for important columns.

In a production version, this dictionary should be frozen after your first validated run and treated as the contract.

In [9]:
# Seeds a minimal schema contract from a live DataFrame: required column list
# and a best-guess semantic type for each column.
def seed_expected_schema(df):
    return {
        "required_columns": list(df.columns),
        "expected_types": {col: infer_semantic_type(df[col]) for col in df.columns}
    }

EXPECTED_SCHEMAS = {name: seed_expected_schema(df) for name, df in datasets.items()}

# Manually override inferred types for columns that are known to be numeric
# but may contain enough formatting noise to fool infer_semantic_type.
for ds_name, df in datasets.items():
    for col in df.columns:
        if col.upper() in {"TOT", "ENTRATI", "INVESTIGATI"} or col.lower() in {"spesa", "attivazioni", "cessazioni"}:
            EXPECTED_SCHEMAS[ds_name]["expected_types"][col] = "numeric"

SCHEMA_NAMING_RULES = {
    "lowercase_only": False,
    "snake_case_preferred": True
}

# Validates a DataFrame's structure against the expected schema:
# checks for missing or unexpected columns, duplicate headers, semantic type
# mismatches, and naming-convention violations.
def check_schema(df, dataset_name, expected_schemas=EXPECTED_SCHEMAS):
    tool = "check_schema"
    issues = []
    expected = expected_schemas.get(dataset_name, {})

    required_columns = expected.get("required_columns", [])
    expected_types = expected.get("expected_types", {})
    actual_columns = list(df.columns)

    missing_required = [c for c in required_columns if c not in actual_columns]
    unexpected_columns = [c for c in actual_columns if c not in required_columns]
    duplicate_columns = df.columns[df.columns.duplicated()].tolist()

    if missing_required:
        issues.append(make_issue(
            dataset_name, tool, "missing_required_columns", "critical",
            f"Missing required columns: {missing_required}",
            columns=missing_required, row_count=len(df),
            evidence={"missing_required": missing_required},
            suggested_fix="Restore missing columns before downstream validation."
        ))

    if unexpected_columns:
        issues.append(make_issue(
            dataset_name, tool, "unexpected_columns", "medium",
            f"Unexpected columns found: {unexpected_columns}",
            columns=unexpected_columns, row_count=len(df),
            evidence={"unexpected_columns": unexpected_columns},
            suggested_fix="Review whether these are valid additions or ingestion artifacts."
        ))

    if duplicate_columns:
        issues.append(make_issue(
            dataset_name, tool, "duplicate_column_names", "high",
            f"Duplicate column names detected: {duplicate_columns}",
            columns=duplicate_columns, row_count=len(df),
            evidence={"duplicate_columns": duplicate_columns},
            suggested_fix="Rename duplicate headers before any transformation or join."
        ))

    actual_types = {col: infer_semantic_type(df[col]) for col in df.columns}
    for col, expected_type in expected_types.items():
        if col in df.columns:
            actual_type = actual_types[col]
            if expected_type != "unknown" and actual_type != "unknown" and actual_type != expected_type:
                issues.append(make_issue(
                    dataset_name, tool, "semantic_type_mismatch", "high",
                    f"Column `{col}` looks like {actual_type}, expected {expected_type}.",
                    columns=[col], row_count=df[col].notna().sum(),
                    evidence={"expected_type": expected_type, "actual_type": actual_type,
                              "samples": safe_samples(df[col])},
                    suggested_fix="Standardize the column representation before agent-level reasoning."
                ))

    for col in df.columns:
        flags = []
        if SCHEMA_NAMING_RULES["snake_case_preferred"]:
            if re.search(r"[\s%-]", col):
                flags.append("special_char_or_space")
        if col[:1].isdigit():
            flags.append("starts_with_digit")
        if flags:
            issues.append(make_issue(
                dataset_name, tool, "naming_convention_violation", "low",
                f"Column `{col}` violates preferred naming conventions.",
                columns=[col], row_count=len(df),
                evidence={"violations": flags},
                suggested_fix="Standardize to a stable naming convention before SQL/API export."
            ))

    return build_result(dataset_name, tool, issues, meta={"column_count": len(df.columns)})

## 3.2 — Completeness Tools

These tools quantify effective missingness, not just pandas nulls.  
They treat placeholder strings such as `N.D.`, `?`, or `-` as missing values because those tokens are semantically empty even when they are not technically null.

In [10]:
MANDATORY_COLUMNS = {
    "spesa":       [c for c in ["rata", "spesa"] if c in spesa.columns],
    "attivazioni": [c for c in ["mese", "anno", "RATA", "attivazioni", "cessazioni"] if c in attivazioni.columns],
    "tipologia":   [c for c in ["DATA_PARTENZA", "ANNO_PARTENZA", "ENTRATI", "INVESTIGATI"] if c in tipologia.columns],
    "allarmi":     [c for c in ["DATA_PARTENZA", "ANNO_PARTENZA", "TOT"] if c in allarmi.columns]
}

# Combines real nulls and disguised null tokens to compute effective missingness per column.
# Severity escalates to critical when mandatory columns exceed 20% effective missingness.
def check_nulls(df, dataset_name, mandatory_columns=MANDATORY_COLUMNS):
    tool = "check_nulls"
    issues = []
    mandatory = mandatory_columns.get(dataset_name, [])

    for col in df.columns:
        total = len(df)
        real_nulls = int(df[col].isna().sum())
        disguised_nulls = int(df[col].astype(str).str.strip().str.lower().isin(DISGUISED_NULLS).sum())
        effective_missing = real_nulls + disguised_nulls
        missing_ratio = effective_missing / total if total else 0

        if effective_missing == 0:
            continue

        if col in mandatory:
            severity = "critical" if missing_ratio >= 0.20 else "high"
            issue_type = "missing_mandatory_values"
        else:
            severity = "medium" if missing_ratio >= 0.20 else "low"
            issue_type = "missing_optional_values"

        issues.append(make_issue(
            dataset_name, tool, issue_type, severity,
            f"Column `{col}` has {effective_missing} effectively missing values ({missing_ratio:.1%}).",
            columns=[col], row_count=effective_missing,
            evidence={"real_nulls": real_nulls, "disguised_nulls": disguised_nulls,
                      "missing_ratio": round(missing_ratio, 4), "samples": safe_samples(df[col])},
            suggested_fix="Impute, standardize, or drop depending on business criticality."
        ))

    return build_result(dataset_name, tool, issues)

# Flags any column whose completeness (share of non-missing values) falls below a
# configurable threshold; useful for quickly surfacing near-empty columns that
# the per-column null check might bury.
def check_sparse_columns(df, dataset_name, threshold=0.85):
    tool = "check_sparse_columns"
    issues = []

    for col in df.columns:
        total = len(df)
        real_nulls = int(df[col].isna().sum())
        disguised_nulls = int(df[col].astype(str).str.strip().str.lower().isin(DISGUISED_NULLS).sum())
        effective_missing = real_nulls + disguised_nulls
        completeness = 1 - (effective_missing / total if total else 0)

        if completeness < threshold:
            severity = "high" if completeness < 0.60 else "medium"
            issues.append(make_issue(
                dataset_name, tool, "sparse_column", severity,
                f"Column `{col}` completeness is only {completeness:.1%}.",
                columns=[col], row_count=effective_missing,
                evidence={"completeness": round(completeness, 4),
                          "real_nulls": real_nulls, "disguised_nulls": disguised_nulls},
                suggested_fix="Assess whether the column should be imputed, deprecated, or excluded."
            ))

    return build_result(dataset_name, tool, issues, meta={"threshold": threshold})

## 3.3 — Format Consistency Tools

A value can be present but still unusable if its format is inconsistent.  
This cell validates temporal fields, month and year encodings, and low-cardinality categorical columns where casing or spelling variants may hide duplicate categories.

In [11]:
FORMAT_RULES = {
    "spesa":       {"rata": "period", "aggregation-time": "datetime"},
    "attivazioni": {"mese": "month", "anno": "year", "RATA": "period", "aggregation-time": "datetime"},
    "tipologia":   {"DATA_PARTENZA": "date", "ANNO_PARTENZA": "year"},
    "allarmi":     {"DATA_PARTENZA": "date", "ANNO_PARTENZA": "year"}
}

# Classifies a single cell value against the expected temporal kind
# (year, month, period, date, or datetime) and returns a label like
# "yyyy-mm-dd", "invalid", or "missing" to build a format distribution.
def classify_format(value, kind):
    if pd.isna(value) or normalize_text(value) is None or is_disguised_null(value):
        return "missing"

    text = str(value).strip()

    if kind == "year":
        return "yyyy" if re.fullmatch(r"(19|20)\d{2}", text) else "invalid"

    if kind == "month":
        return "m_or_mm" if re.fullmatch(r"(0?[1-9]|1[0-2])", text) else "invalid"

    if kind == "period":
        if re.fullmatch(r"\d{4}-\d{2}", text): return "yyyy-mm"
        if re.fullmatch(r"\d{2}/\d{4}", text): return "mm/yyyy"
        if re.fullmatch(r"\d{4}/\d{2}", text): return "yyyy/mm"
        return "invalid"

    if kind == "date":
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}", text): return "yyyy-mm-dd"
        if re.fullmatch(r"\d{2}/\d{2}/\d{4}", text): return "dd/mm/yyyy"
        if pd.to_datetime(pd.Series([text]), errors="coerce", dayfirst=True).notna().iloc[0]:
            return "other_parseable_date"
        return "invalid"

    if kind == "datetime":
        if pd.to_datetime(pd.Series([text]), errors="coerce", dayfirst=True).notna().iloc[0]:
            return "datetime_like" if ("T" in text or ":" in text) else "date_like"
        return "invalid"

    return "unknown"

# Scans columns listed in FORMAT_RULES and raises an issue when invalid values
# exist or when multiple valid format families are mixed in the same column
# (e.g., some dates as "dd/mm/yyyy" and others as "yyyy-mm-dd").
def check_formats(df, dataset_name, format_rules=FORMAT_RULES):
    tool = "check_formats"
    issues = []
    rules = format_rules.get(dataset_name, {})

    for col, kind in rules.items():
        if col not in df.columns:
            continue

        classified = df[col].apply(lambda x: classify_format(x, kind))
        non_missing = classified[classified != "missing"]

        if len(non_missing) == 0:
            continue

        invalid_count = int((non_missing == "invalid").sum())
        valid_share = 1 - (invalid_count / len(non_missing))
        format_mix = non_missing[non_missing != "invalid"].value_counts().to_dict()

        if invalid_count > 0:
            severity = "high" if valid_share < 0.80 else "medium"
            issues.append(make_issue(
                dataset_name, tool, "invalid_format_values", severity,
                f"Column `{col}` contains {invalid_count} invalid {kind} values.",
                columns=[col], row_count=invalid_count,
                evidence={"kind": kind, "valid_share": round(valid_share, 4),
                          "format_distribution": format_mix, "samples": safe_samples(df[col])},
                suggested_fix="Normalize this field to one accepted format before validation or joins."
            ))

        if len(format_mix) > 1:
            issues.append(make_issue(
                dataset_name, tool, "mixed_format_family", "medium",
                f"Column `{col}` mixes multiple valid format families: {list(format_mix.keys())}.",
                columns=[col], row_count=len(non_missing),
                evidence={"kind": kind, "format_distribution": format_mix},
                suggested_fix="Convert all values to a single canonical format."
            ))

    return build_result(dataset_name, tool, issues)

# Looks for casing or spelling variants that map to the same lowercase token
# (e.g., "Roma", "ROMA", "roma") in low-cardinality string columns, which would
# inflate group counts in any downstream aggregation.
def check_categorical_case_variants(df, dataset_name, max_unique=40):
    tool = "check_categorical_case_variants"
    issues = []
    object_cols = df.select_dtypes(include="object").columns.tolist()

    for col in object_cols:
        s = df[col].dropna().astype(str).str.strip()
        s = s[s.ne("")]
        if len(s) == 0 or s.nunique() > max_unique:
            continue

        grouped = defaultdict(set)
        for val in s.unique():
            grouped[val.lower()].add(val)

        suspicious = {k: sorted(list(v)) for k, v in grouped.items() if len(v) > 1}
        if suspicious:
            issues.append(make_issue(
                dataset_name, tool, "categorical_variant_inconsistency", "low",
                f"Column `{col}` contains casing or spelling variants that may represent the same category.",
                columns=[col],
                row_count=sum(len(v) for v in suspicious.values()),
                evidence={"variant_groups": suspicious},
                suggested_fix="Standardize category labels before aggregation or modeling."
            ))

    return build_result(dataset_name, tool, issues)

## 3.4 — Numeric Validity, Outliers, and Duplicates

This block separates structural numeric problems from statistical anomalies.  
First we detect invalid numeric encodings and impossible values; then we use IQR (interquartile range) to flag extreme observations; finally we check exact duplicate rows and optional duplicate business keys.

In [12]:
NUMERIC_RULES = {
    "spesa":       {"spesa":        {"min": 0, "forbid_tokens": ["€"]}},
    "attivazioni": {"attivazioni":  {"min": 0}, "cessazioni": {"min": 0}},
    "tipologia":   {"ENTRATI":      {"min": 0}, "INVESTIGATI": {"min": 0}},
    "allarmi":     {"TOT":          {"min": 0}}
}

DUPLICATE_KEY_RULES = {
    "spesa": [], "attivazioni": [], "tipologia": [], "allarmi": []
}

# Validates numeric columns by detecting values that cannot be parsed to a number
# even after loose cleaning, forbidden tokens like "€", and values that violate
# a known minimum threshold (e.g., negative counts or costs).
def check_numeric_validity(df, dataset_name, numeric_rules=NUMERIC_RULES):
    tool = "check_numeric_validity"
    issues = []
    rules = numeric_rules.get(dataset_name, {})

    for col, rule in rules.items():
        if col not in df.columns:
            continue

        raw = df[col]
        raw_text = raw.astype(str).str.strip()
        parsed = coerce_numeric_loose(raw)
        present_mask = raw.notna() & raw_text.ne("") & ~raw_text.str.lower().isin(DISGUISED_NULLS)
        invalid_mask = present_mask & parsed.isna()

        if invalid_mask.any():
            issues.append(make_issue(
                dataset_name, tool, "non_numeric_values_in_numeric_field", "high",
                f"Column `{col}` contains non-numeric values in a numeric field.",
                columns=[col], row_count=int(invalid_mask.sum()),
                evidence={"samples": raw[invalid_mask].astype(str).drop_duplicates().head(10).tolist()},
                suggested_fix="Strip symbols/text and coerce to numeric using a canonical parser."
            ))

        for token in rule.get("forbid_tokens", []):
            token_mask = raw.astype(str).str.contains(re.escape(token), na=False)
            if token_mask.any():
                issues.append(make_issue(
                    dataset_name, tool, "forbidden_token_in_numeric_field", "medium",
                    f"Column `{col}` contains forbidden token `{token}`.",
                    columns=[col], row_count=int(token_mask.sum()),
                    evidence={"token": token, "samples": raw[token_mask].astype(str).drop_duplicates().head(10).tolist()},
                    suggested_fix="Store numeric values without currency/unit symbols."
                ))

        min_value = rule.get("min")
        if min_value is not None:
            negative_mask = parsed.notna() & (parsed < min_value)
            if negative_mask.any():
                issues.append(make_issue(
                    dataset_name, tool, "value_below_minimum", "high",
                    f"Column `{col}` contains values below the allowed minimum ({min_value}).",
                    columns=[col], row_count=int(negative_mask.sum()),
                    evidence={"min_allowed": min_value,
                              "samples": raw[negative_mask].astype(str).drop_duplicates().head(10).tolist()},
                    suggested_fix="Review source data or clamp/remove impossible values."
                ))

    return build_result(dataset_name, tool, issues)

# Uses the Interquartile Range (IQR) method to flag statistical outliers in numeric
# columns. The whisker multiplier (default 1.5) controls sensitivity; only columns
# with enough data variety are tested to avoid false positives on near-constant fields.
def check_outliers_iqr(df, dataset_name, numeric_rules=NUMERIC_RULES, whisker=1.5):
    tool = "check_outliers_iqr"
    issues = []
    candidate_cols = [c for c in numeric_rules.get(dataset_name, {}) if c in df.columns]

    for col in candidate_cols:
        vals = coerce_numeric_loose(df[col]).dropna()
        if len(vals) < 8 or vals.nunique() < 4:
            continue

        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue

        lower, upper = q1 - whisker * iqr, q3 + whisker * iqr
        outlier_mask = coerce_numeric_loose(df[col]).notna() & ~coerce_numeric_loose(df[col]).between(lower, upper)
        outlier_count = int(outlier_mask.sum())

        if outlier_count > 0:
            severity = "medium" if outlier_count / len(df) < 0.05 else "high"
            issues.append(make_issue(
                dataset_name, tool, "iqr_outliers", severity,
                f"Column `{col}` contains {outlier_count} IQR-based outliers.",
                columns=[col], row_count=outlier_count,
                evidence={"q1": float(q1), "q3": float(q3), "iqr": float(iqr),
                          "lower_bound": float(lower), "upper_bound": float(upper),
                          "samples": df.loc[outlier_mask, col].astype(str).drop_duplicates().head(10).tolist()},
                suggested_fix="Review whether these are valid spikes or ingestion errors."
            ))

    return build_result(dataset_name, tool, issues, meta={"whisker": whisker})

# Detects fully identical duplicate rows (exact copies) and, when business-key
# columns are configured, also finds key duplicates that carry divergent data —
# a more dangerous form of duplication that silently inflates aggregations.
def check_duplicates(df, dataset_name, duplicate_key_rules=DUPLICATE_KEY_RULES):
    tool = "check_duplicates"
    issues = []

    exact_dup_mask = df.duplicated(keep=False)
    if exact_dup_mask.any():
        issues.append(make_issue(
            dataset_name, tool, "exact_duplicate_rows", "medium",
            "Exact duplicate rows detected.",
            columns=list(df.columns), row_count=int(exact_dup_mask.sum()),
            evidence={"sample_rows": df[exact_dup_mask].head(5).to_dict(orient="records")},
            suggested_fix="Deduplicate exact copies before aggregation or scoring."
        ))

    key_cols = [c for c in duplicate_key_rules.get(dataset_name, []) if c in df.columns]
    if key_cols:
        key_dup_mask = df.duplicated(subset=key_cols, keep=False)
        if key_dup_mask.any():
            issues.append(make_issue(
                dataset_name, tool, "duplicate_business_keys", "high",
                f"Duplicate business keys detected on {key_cols}.",
                columns=key_cols, row_count=int(key_dup_mask.sum()),
                evidence={"sample_rows": df[key_dup_mask].head(5).to_dict(orient="records")},
                suggested_fix="Investigate whether repeated keys represent conflicts or valid snapshots."
            ))

    return build_result(dataset_name, tool, issues)

## 3.5 — Cross-Column Logic Rules

Some errors only appear when columns are evaluated together.  
This tool enforces dataset-specific business logic such as year/date agreement, month-year alignment, and logical count constraints like `INVESTIGATI <= ENTRATI`.

In [13]:
# Parses a string that should contain a 4-digit year; returns np.nan on failure.
def parse_year(text):
    if pd.isna(text) or normalize_text(text) is None or is_disguised_null(text):
        return np.nan
    text = str(text).strip()
    return int(text) if re.fullmatch(r"(19|20)\d{2}", text) else np.nan

# Parses a period string (yyyy-mm, mm/yyyy, yyyy/mm) into a (year, month) tuple;
# returns (nan, nan) when the format is unrecognized.
def parse_year_month(text):
    if pd.isna(text) or normalize_text(text) is None or is_disguised_null(text):
        return (np.nan, np.nan)
    text = str(text).strip()
    if re.fullmatch(r"\d{4}-\d{2}", text):
        y, m = text.split("-"); return int(y), int(m)
    if re.fullmatch(r"\d{4}/\d{2}", text):
        y, m = text.split("/"); return int(y), int(m)
    if re.fullmatch(r"\d{2}/\d{4}", text):
        m, y = text.split("/"); return int(y), int(m)
    return (np.nan, np.nan)

# Attempts to parse a date string and extracts only the year component;
# returns np.nan when the string cannot be parsed as a date.
def parse_date_year(text):
    if pd.isna(text) or normalize_text(text) is None or is_disguised_null(text):
        return np.nan
    dt = pd.to_datetime(pd.Series([str(text).strip()]), errors="coerce", dayfirst=True).iloc[0]
    return dt.year if pd.notna(dt) else np.nan

# Enforces dataset-specific multi-column business rules:
#   - tipologia:   INVESTIGATI must not exceed ENTRATI (logical count ceiling)
#   - tipologia/allarmi: year extracted from DATA_PARTENZA must match ANNO_PARTENZA
#   - attivazioni: mese and anno must be consistent with the RATA period string
def check_cross_column(df, dataset_name):
    tool = "check_cross_column"
    issues = []

    if dataset_name == "tipologia":
        if {"ENTRATI", "INVESTIGATI"}.issubset(df.columns):
            entrati = coerce_numeric_loose(df["ENTRATI"])
            investigati = coerce_numeric_loose(df["INVESTIGATI"])
            bad = investigati.notna() & entrati.notna() & (investigati > entrati)
            if bad.any():
                issues.append(make_issue(
                    dataset_name, tool, "logical_count_violation", "critical",
                    "`INVESTIGATI` cannot be greater than `ENTRATI`.",
                    columns=["INVESTIGATI", "ENTRATI"], row_count=int(bad.sum()),
                    evidence={"sample_rows": df.loc[bad, ["INVESTIGATI", "ENTRATI"]].head(10).to_dict(orient="records")},
                    suggested_fix="Review row-level counts and restore logical consistency."
                ))

    if dataset_name in {"tipologia", "allarmi"}:
        if {"DATA_PARTENZA", "ANNO_PARTENZA"}.issubset(df.columns):
            data_year = df["DATA_PARTENZA"].apply(parse_date_year)
            anno = df["ANNO_PARTENZA"].apply(parse_year)
            bad = data_year.notna() & anno.notna() & (data_year != anno)
            if bad.any():
                issues.append(make_issue(
                    dataset_name, tool, "year_date_mismatch", "high",
                    "`ANNO_PARTENZA` does not match the year extracted from `DATA_PARTENZA`.",
                    columns=["DATA_PARTENZA", "ANNO_PARTENZA"], row_count=int(bad.sum()),
                    evidence={"sample_rows": df.loc[bad, ["DATA_PARTENZA", "ANNO_PARTENZA"]].head(10).to_dict(orient="records")},
                    suggested_fix="Derive one field from the other or standardize both from source."
                ))

    if dataset_name == "attivazioni":
        if {"mese", "anno", "RATA"}.issubset(df.columns):
            ym_from_rata = df["RATA"].apply(parse_year_month)
            rata_year  = ym_from_rata.apply(lambda x: x[0])
            rata_month = ym_from_rata.apply(lambda x: x[1])
            anno = df["anno"].apply(parse_year)
            mese = pd.to_numeric(df["mese"], errors="coerce")
            bad = anno.notna() & mese.notna() & rata_year.notna() & rata_month.notna() & (
                (anno != rata_year) | (mese != rata_month)
            )
            if bad.any():
                issues.append(make_issue(
                    dataset_name, tool, "month_year_period_mismatch", "high",
                    "`mese` and `anno` are inconsistent with `RATA`.",
                    columns=["mese", "anno", "RATA"], row_count=int(bad.sum()),
                    evidence={"sample_rows": df.loc[bad, ["mese", "anno", "RATA"]].head(10).to_dict(orient="records")},
                    suggested_fix="Rebuild the period fields from one canonical temporal source."
                ))

    return build_result(dataset_name, tool, issues)

## 3.6 — Tool Registry and Dataset Runner

The last step, wires all deterministic tools into one execution pipeline.  
This produces two outputs: a nested dictionary for agent consumption and a flat issues table for quick human inspection.

In [14]:
PHASE3_TOOLS = [
    check_schema,
    check_nulls,
    check_sparse_columns,
    check_formats,
    check_categorical_case_variants,
    check_numeric_validity,
    check_outliers_iqr,
    check_duplicates,
    check_cross_column
]

# Keyed by function name so LangGraph agents can call tools by string reference.
agent_tools = {tool.__name__: tool for tool in PHASE3_TOOLS}

# Runs every tool in PHASE3_TOOLS against a single DataFrame and collects their results.
def audit_dataset(df, dataset_name):
    return {
        "dataset": dataset_name,
        "results": [tool(df, dataset_name) for tool in PHASE3_TOOLS]
    }

# Explodes all nested issue dicts across all datasets into a single flat DataFrame
# for easy sorting, filtering, and display.
def flatten_issues(audit_output):
    rows = []
    for dataset_name, payload in audit_output.items():
        for result in payload["results"]:
            rows.extend(result["issues"])
    return pd.DataFrame(rows)

# Produces a one-row-per-tool summary table showing issue counts and
# severity breakdowns — useful as a quick health check before agent reasoning.
def flatten_summary(audit_output):
    rows = []
    for dataset_name, payload in audit_output.items():
        for result in payload["results"]:
            rows.append({
                "dataset": dataset_name,
                "tool": result["tool"],
                "issue_count": result["issue_count"],
                "severity_breakdown": json.dumps(result["severity_breakdown"]),
            })
    return pd.DataFrame(rows)

phase3_results = {name: audit_dataset(df, name) for name, df in datasets.items()}
phase3_issues  = flatten_issues(phase3_results)
phase3_summary = flatten_summary(phase3_results)

print("Phase 3 completed.")
print(f"Total issues detected: {len(phase3_issues)}")

display(phase3_summary.sort_values(["dataset", "tool"]).reset_index(drop=True))

if not phase3_issues.empty:
    display(
        phase3_issues[
            ["dataset", "tool", "issue_type", "severity", "columns", "row_count", "message"]
        ].sort_values(
            by=["dataset", "severity"],
            key=lambda s: s.map(SEVERITY_RANK) if s.name == "severity" else s,
            ascending=[True, False]
        ).reset_index(drop=True)
    )

Phase 3 completed.
Total issues detected: 117


,dataset,tool,issue_count,severity_breakdown
0,allarmi,check_categorical_case_variants,0,{}
1,allarmi,check_cross_column,1,"{""high"": 1}"
2,allarmi,check_duplicates,1,"{""medium"": 1}"
3,allarmi,check_formats,2,"{""medium"": 2}"
4,allarmi,check_nulls,9,"{""low"": 5, ""high"": 1, ""medium"": 3}"
5,allarmi,check_numeric_validity,1,"{""high"": 1}"
6,allarmi,check_outliers_iqr,1,"{""high"": 1}"
7,allarmi,check_schema,5,"{""low"": 5}"
8,allarmi,check_sparse_columns,3,"{""medium"": 1, ""high"": 2}"
9,attivazioni,check_categorical_case_variants,0,{}


,dataset,tool,issue_type,severity,columns,row_count,message
0,allarmi,check_nulls,missing_mandatory_values,high,[TOT],75,Column `TOT` has 75 effectively missing values...
1,allarmi,check_sparse_columns,sparse_column,high,[note_operatore],5003,Column `note_operatore` completeness is only 1...
2,allarmi,check_sparse_columns,sparse_column,high,[flag_rischio],5029,Column `flag_rischio` completeness is only 1.0%.
3,allarmi,check_numeric_validity,value_below_minimum,high,[TOT],7,Column `TOT` contains values below the allowed...
4,allarmi,check_outliers_iqr,iqr_outliers,high,[TOT],1039,Column `TOT` contains 1039 IQR-based outliers.
...,...,...,...,...,...,...,...
112,tipologia,check_nulls,missing_optional_values,low,[NUMERO_VOLO],204,Column `NUMERO_VOLO` has 204 effectively missi...
113,tipologia,check_nulls,missing_optional_values,low,[FASCIA ETA],523,Column `FASCIA ETA` has 523 effectively missin...
114,tipologia,check_categorical_case_variants,categorical_variant_inconsistency,low,[TIPO_DOCUMENTO],2,Column `TIPO_DOCUMENTO` contains casing or spe...
115,tipologia,check_categorical_case_variants,categorical_variant_inconsistency,low,[GENERE],10,Column `GENERE` contains casing or spelling va...


## Phase 4 — Synthetic Benchmark Dataset

The deterministic tool library built in Phase 3 requires a quantitative validation before it can be trusted as evidence for agent reasoning. This phase constructs that validation by producing a controlled benchmark with a known, auditable ground truth.

The process runs in three steps. First, a clean baseline is sampled from each dataset — rows with effective nulls in mandatory columns are excluded so the baseline contains zero pre-existing anomalies. Second, a fixed set of synthetic errors is injected at randomly selected positions: disguised nulls, non-numeric encodings, negative values, IQR outliers, malformed dates, exact duplicate rows, cross-column logical violations, and categorical casing variants. Every injection is recorded in a registry that stores the affected dataset, column, row index, error type, original value, and injected value. Third, the full Phase 3 audit is executed against the corrupted copies and its detections are matched against the registry.

Matching operates at the (dataset, error\_type) level. A detection is a **True Positive** when a tool's `issue_type` maps to an injected `error_type` in the same dataset, a **False Positive** when it fires on a pair absent from the registry, and a **False Negative** when an injected pair goes undetected. From these counts the phase derives Precision, Recall, and F1 globally and per error type.

\[
\text{Precision} = \frac{TP}{TP + FP}
\qquad
\text{Recall} = \frac{TP}{TP + FN}
\qquad
F_1 = 2 \cdot \frac{P \cdot R}{P + R}
\]

A high F1 at this stage confirms that the deterministic layer is reliable in isolation. Any residual misses in Phase 5–6 can therefore be attributed to the LLM reasoning layer rather than to the tools it calls.

All artifacts — ground truth registry, detected events table, evaluation metrics, corrupted CSVs, and the detection heatmap — are written to `data/benchmark/` and consumed directly by Phase 7 (Reliability Scoring) and Phase 9 (Streamlit dashboard).